In [12]:
# 1. 导入所有依赖库
import os
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np

# 导入我们的核心模块
from models.apm_former import APM_Former_ImageOnly
from utils.dataset import get_adni_dataloaders
# 修正导入：只导入config里存在的变量
from utils.config import TRAIN_CSV, VAL_CSV, BATCH_SIZE, NUM_WORKERS, TRAIN_IMG_SIZE
from utils.logging_utils import get_logger
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR


# 日志
logger = get_logger("Train")

In [13]:
# 2. 超参数配置
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LEARNING_RATE = 5e-4
warmup_epochs = 5  # 前 5 个 epoch 用来预热
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 30
GRADIENT_ACCUMULATION_STEPS = 1
NUM_CLASSES = 2
FEATURE_SIZE = 48
GUIDE_CHANNELS = 18
SAVE_PATH = "checkpoints/best_model.pth"
PRETRAINED_SWIN_PATH = "checkpoints/model_swinvit.pt"

logger.info(f"训练设备: {DEVICE}")
logger.info(f"批次大小: {BATCH_SIZE}")
logger.info(f"总轮数: {NUM_EPOCHS}")

2026-04-22 08:27:11,435 - Train - INFO - 训练设备: cuda
2026-04-22 08:27:11,438 - Train - INFO - 批次大小: 4


2026-04-22 08:27:11,439 - Train - INFO - 总轮数: 30


In [14]:
# 3. 加载训练集 + 验证集 DataLoader
logger.info("正在加载数据集...")
train_loader, val_loader = get_adni_dataloaders(
    train_csv=TRAIN_CSV,
    val_csv=VAL_CSV,
    batch_size=BATCH_SIZE,
    target_size=TRAIN_IMG_SIZE,
    num_workers=NUM_WORKERS
)

2026-04-22 08:27:11,468 - Train - INFO - 正在加载数据集...


2026-04-22 08:27:11,561 - utils.dataset - INFO - ✅ 成功加载 659 个样本
2026-04-22 08:27:11,570 - utils.dataset - INFO - ✅ 成功加载 82 个样本
2026-04-22 08:27:11,572 - utils.dataset - INFO - ✅ DataLoader 构建完成！
2026-04-22 08:27:11,573 - utils.dataset - INFO - 训练集批次: 165，验证集批次: 21


In [15]:
# 4. 初始化模型（完整版！）
logger.info("初始化 APM-Former 模型...")
model = APM_Former_ImageOnly(
    img_size=TRAIN_IMG_SIZE,
    in_channels=1,
    num_classes=NUM_CLASSES,
    feature_size=FEATURE_SIZE,
    guide_channels=GUIDE_CHANNELS,
    pretrained_swin_path=PRETRAINED_SWIN_PATH # 传入刚刚下载的权重
).to(DEVICE)

# 打印模型参数量
total_params = sum(p.numel() for p in model.parameters())
logger.info(f"模型总参数量: {total_params / 1e6:.2f} M")

2026-04-22 08:27:11,614 - Train - INFO - 初始化 APM-Former 模型...
/home/ubuntu/Code/APM_Former_Project/models/apm_former.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  che

加载SwinUNETR预训练权重：checkpoints/model_swinvit.pt


2026-04-22 08:27:13,129 - Train - INFO - 模型总参数量: 64.39 M


In [16]:
# # 5. 损失函数 + 优化器
# class FocalLoss(nn.Module):
#     def __init__(self, alpha=0.25, gamma=2, reduction='mean'):
#         super().__init__()
#         self.alpha = alpha
#         self.gamma = gamma
#         self.ce = nn.CrossEntropyLoss()

#     def forward(self, inputs, targets):
#         ce_loss = self.ce(inputs, targets)
#         pt = torch.exp(-ce_loss)
#         focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
#         return focal_loss.mean()

# criterion = FocalLoss(alpha=0.36, gamma=2).to(DEVICE) # alpha 可以设为验证集正样本的比例 0.36，或保持默认 0.25# criterion = nn.CrossEntropyLoss().to(DEVICE)
from monai.losses import FocalLoss

# 计算类别权重：训练集 0类有421例，1类有238例
# weight_pMCI = 421 / 238 ≈ 1.768
weights = torch.tensor([1.0, 1.768]).to(DEVICE) 

# 使用 MONAI 的 FocalLoss，to_onehot_y=True 是必须的，因为你的 label 是类别索引
criterion = FocalLoss(weight=weights, gamma=2.0, to_onehot_y=True).to(DEVICE)


optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# 2. 引入带 Warmup 的余弦退火学习率调度器

# 预热阶段：学习率从 0.01 * 5e-4 逐渐增长到 5e-4
scheduler_warmup = LinearLR(optimizer, start_factor=0.01, total_iters=warmup_epochs)
# 余弦退火阶段：在剩下的 25 个 epoch 里衰减
scheduler_cosine = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - warmup_epochs)

# 组合调度器
scheduler = SequentialLR(
    optimizer, 
    schedulers=[scheduler_warmup, scheduler_cosine], 
    milestones=[warmup_epochs]
)


In [17]:
# 6. 训练/验证函数
def train_epoch(model, loader, criterion, optimizer, device, accum_steps):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    optimizer.zero_grad()
    for idx, (images, labels) in enumerate(tqdm(loader, desc="训练中")):
        images = images.to(device)
        labels = labels.to(device)

        logits, _, _ = model(images)
        loss = criterion(logits, labels.unsqueeze(1))
        loss = loss / accum_steps

        loss.backward()

        if (idx + 1) % accum_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = 100 * correct / total
    return avg_loss, acc

@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="验证中"):
        images = images.to(device)
        labels = labels.to(device)

        logits, _, _ = model(images)
        loss = criterion(logits, labels.unsqueeze(1))

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = 100 * correct / total
    return avg_loss, acc

In [18]:
# %% 排查：查看训练集、验证集的标签分布
import pandas as pd
from utils.config import TRAIN_CSV, VAL_CSV

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

print("===== 训练集标签分布 =====")
print(train_df['Label'].value_counts())
print("比例：", train_df['Label'].value_counts(normalize=True))

print("\n===== 验证集标签分布 =====")
print(val_df['Label'].value_counts())
print("比例：", val_df['Label'].value_counts(normalize=True))

===== 训练集标签分布 =====
Label
0    421
1    238
Name: count, dtype: int64
比例： Label
0    0.638847
1    0.361153
Name: proportion, dtype: float64

===== 验证集标签分布 =====
Label
0    52
1    30
Name: count, dtype: int64
比例： Label
0    0.634146
1    0.365854
Name: proportion, dtype: float64


In [19]:
# %% 排查：查看 DataLoader 中的标签统计
# from collections import Counter

# train_labels = []
# for imgs, lbls in train_loader:
#     train_labels.extend(lbls.numpy().tolist())

# print("训练集加载到的标签统计：", Counter(train_labels))

In [20]:
# 强制给模型看标签1，看是否输出正常
model.eval()
test_img = torch.randn(1,1,96,96,96).to(DEVICE)
with torch.no_grad():
    logits,_,_ = model(test_img)
    print("模型输出概率：", torch.softmax(logits, dim=1))

模型输出概率： tensor([[0.5511, 0.4489]], device='cuda:0')


In [21]:
# %% 训练1个batch，打印梯度！！！
model.train()
images, labels = next(iter(train_loader))
images = images.to(DEVICE)
labels = labels.to(DEVICE)

optimizer.zero_grad()
logits, _, _ = model(images)
loss = criterion(logits, labels)
loss.backward()

# 打印Swin主干的梯度（如果是None/0，就是主干冻结！）
print("Loss:", loss.item())
print("Swin主干参数梯度：")
for name, param in model.swin_backbone.named_parameters():
    if param.grad is not None:
        print(f"{name}: 梯度均值 = {param.grad.mean().item()}")
        break
    else:
        print(f"{name}: 梯度为 None ❌（冻结了！）")
        break

Loss: 0.22727853059768677
Swin主干参数梯度：
swinViT.patch_embed.proj.weight: 梯度均值 = 6.969300443415705e-07


In [22]:
# 7. 主训练循环
best_val_acc = 0.0
logger.info("=" * 50)
logger.info("开始训练！")
logger.info("=" * 50)

for epoch in range(NUM_EPOCHS):
    logger.info(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE, GRADIENT_ACCUMULATION_STEPS)
    val_loss, val_acc = val_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()

    logger.info(f"训练损失: {train_loss:.4f} | 训练准确率: {train_acc:.2f}%")
    logger.info(f"验证损失: {val_loss:.4f} | 验证准确率: {val_acc:.2f}%")
    logger.info(f"当前学习率: {optimizer.param_groups[0]['lr']:.8f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        logger.info(f"✅ 最佳模型已保存！最佳验证准确率: {best_val_acc:.2f}%")

logger.info("\n🎉 训练完成！")
logger.info(f"最终最佳验证准确率: {best_val_acc:.2f}%")

2026-04-22 08:27:17,695 - Train - INFO - ==================================================
2026-04-22 08:27:17,697 - Train - INFO - 开始训练！
2026-04-22 08:27:17,699 - Train - INFO - ==================================================
2026-04-22 08:27:17,700 - Train - INFO - 
Epoch [1/30]
验证中: 100%|██████████| 21/21 [00:07<00:00,  2.65it/s]
2026-04-22 08:28:37,843 - Train - INFO - 训练损失: 0.2301 | 训练准确率: 63.88%
2026-04-22 08:28:37,844 - Train - INFO - 验证损失: 0.2273 | 验证准确率: 63.41%
2026-04-22 08:28:37,845 - Train - INFO - 当前学习率: 0.00010400
2026-04-22 08:28:39,219 - Train - INFO - ✅ 最佳模型已保存！最佳验证准确率: 63.41%
2026-04-22 08:28:39,220 - Train - INFO - 
Epoch [2/30]
验证中: 100%|██████████| 21/21 [00:09<00:00,  2.14it/s]
2026-04-22 08:30:09,096 - Train - INFO - 训练损失: 0.2309 | 训练准确率: 63.88%
2026-04-22 08:30:09,097 - Train - INFO - 验证损失: 0.2284 | 验证准确率: 63.41%
2026-04-22 08:30:09,098 - Train - INFO - 当前学习率: 0.00020300
2026-04-22 08:30:09,099 - Train - INFO - 
Epoch [3/30]
验证中: 100%|██████████| 21/21 [00:0